# Terry's Resident Engineer / Field Inspector Voice — Free QLoRA Fine-Tuning (Google Colab)

**Cost:** `$0.00` on Google Colab Free T4 GPU.

### Before clicking `Runtime -> Run all`:
1. Go to **Runtime → Change runtime type** and confirm **Hardware accelerator = T4 GPU**.
2. Keep these two files ready from your PC folder `C:\Users\Terry\OneDrive\daily report upgrade final\DAILY-REPORTER-V3\training_data\`:
   - `inspector_voice_train.jsonl`
   - `inspector_voice_val.jsonl`
3. Run the cells in order. Cell 2 prompts you to select both `.jsonl` files.

In [ ]:
# 1. Install Unsloth & QLoRA packages ($0 Colab T4)
%%capture
!pip install --upgrade -qqq uv
!uv pip install --system -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" "trl>=0.15.0" peft accelerate bitsandbytes datasets

In [ ]:
# 2. Upload inspector_voice_train.jsonl and inspector_voice_val.jsonl from your PC
import json
import pathlib
from google.colab import files

uploaded = files.upload()

for required in ("inspector_voice_train.jsonl", "inspector_voice_val.jsonl"):
    p = pathlib.Path(required)
    assert p.exists(), f"Missing {required} — please upload both .jsonl files."
    lines = p.read_text(encoding="utf-8").splitlines()
    first = json.loads(lines[0])
    assert "messages" in first, f"Invalid JSONL format in {required}"
    print(f"Verified {required}: {len(lines)} examples")

In [ ]:
# 3. Load Base Model + Attach LoRA Adapter
# Qwen2.5-7B-Instruct exports to a ~4.7 GB Q4_K_M GGUF that fits 100% in your RTX 5060 8 GB VRAM.
# If training solely for cloud serverless LoRA (Fireworks / Together), you can also use:
#   BASE_MODEL = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 8192

print(f"GPU: {torch.cuda.get_device_name(0)}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA adapter attached and ready.")

In [ ]:
# 4. Format Dataset & Train (3 epochs across all 1,267 MCN examples — zero truncation)
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

raw_datasets = load_dataset(
    "json",
    data_files={
        "train": "inspector_voice_train.jsonl",
        "validation": "inspector_voice_val.jsonl",
    },
)

def format_prompts(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(
                convo, tokenize=False, add_generation_prompt=False
            )
            for convo in batch["messages"]
        ]
    }

formatted = raw_datasets.map(format_prompts, batched=True)
print(f"Train: {len(formatted['train'])} | Val: {len(formatted['validation'])}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted["train"],
    eval_dataset=formatted["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=15,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="checkpoints",
        report_to="none",
    ),
)

stats = trainer.train()
print("Training complete:", stats.metrics)

In [ ]:
# 5. Sanity Check — Verify Voice Before Exporting
FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": (
            "You are a 20+ year Pipeline Construction Resident Engineer rewriting draft "
            "daily report bullets into your own direct, plain field-inspector voice. "
            "Write zero parentheses anywhere in the output."
        ),
    },
    {
        "role": "user",
        "content": (
            "DRAFT FIELD TEXT TO REWRITE IN RESIDENT ENGINEER VOICE:\n"
            "• The inspector visited Blow Off No. 9 prior to PUD arriving and met with Eduardo to identify items needing remediation.\n"
            "• HMS commenced cutting seven (7) traffic loops at Genesee & Decoro by 12:00 AM (midnight) and installed Best Management Practices (BMPs).\n"
            "• AR Concrete compacted subgrade utilizing a jumping jack tamper and vibratory plate compactor."
        ),
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1)
reply = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("=== FINE-TUNED MODEL OUTPUT ===\n" + reply)

In [ ]:
# 6. Save LoRA Adapter (~160 MB) & Download ZIP (for Fireworks / Together Cloud Hosting)
import shutil
from google.colab import files

model.save_pretrained("terry-inspector-lora")
tokenizer.save_pretrained("terry-inspector-lora")
shutil.make_archive("terry-inspector-lora", "zip", "terry-inspector-lora")
print("Saved terry-inspector-lora.zip")
files.download("terry-inspector-lora.zip")

In [ ]:
# 7. Export Q4_K_M GGUF (~4.7 GB) Directly to Your Google Drive for Local Ollama ($0 on PC)
# Mounting Google Drive avoids browser timeouts when transferring the 4.7 GB GGUF file.
import pathlib
import shutil
from google.colab import drive

drive.mount("/content/drive")
drive_out = pathlib.Path("/content/drive/MyDrive/Terry_Inspector_Model")
drive_out.mkdir(parents=True, exist_ok=True)

model.save_pretrained_gguf(
    "terry-inspector-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

for gguf_file in pathlib.Path("terry-inspector-gguf").glob("*.gguf"):
    dest = drive_out / "terry-inspector-q4_k_m.gguf"
    shutil.copy2(gguf_file, dest)
    print(f"Copied {dest} ({dest.stat().st_size / (1024**3):.2f} GB)")

modelfile_text = (
    'FROM ./terry-inspector-q4_k_m.gguf\n\n'
    'PARAMETER temperature 0.1\n'
    'PARAMETER top_p 0.9\n'
    'PARAMETER num_ctx 8192\n\n'
    'SYSTEM """You are a 20+ year Pipeline Construction Resident Engineer and Field Inspector '
    'writing your own daily construction report in plain, direct field English. Keep "I" or direct '
    'active past-tense verbs when describing your own inspections, site visits, meetings, or '
    'directions to a contractor. Name the contractor or "The crew" for crew work. Write zero '
    'parentheses anywhere in prose or bullets. Write counts once as plain digits and bare standard '
    'acronyms."""\n'
)
(drive_out / "Modelfile").write_text(modelfile_text, encoding="utf-8")
shutil.copy2("terry-inspector-lora.zip", drive_out / "terry-inspector-lora.zip")
print("All files saved to Google Drive -> MyDrive/Terry_Inspector_Model/")
print("On your PC, open PowerShell in that folder and run:")
print("  ollama create terry-inspector -f Modelfile")